# 00 · Setup check and T4 test
Run this **once in every Colab account** you will use.

Before running: `Runtime -> Change runtime type -> T4 GPU -> Save`. Then `Runtime -> Run all`.

It checks four things: the shared Drive folder is visible, the GPU works, the whole pipeline runs
end to end on a small synthetic dataset (about 2-4 min, nothing is written to the real data), and how long
one model fit takes on the T4.

In [ ]:
# ---- setup: identical in every notebook ----
FOLDER = "AMR_AL"          # name of the shared folder (or its shortcut) in My Drive

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
ROOT = f"/content/drive/MyDrive/{FOLDER}"
if not os.path.isdir(os.path.join(ROOT, "amr_al")):
    raise SystemExit(f"{ROOT}/amr_al not found.\nIn Google Drive open 'Shared with me', right-click {FOLDER} -> "
                     "Organize -> Add shortcut -> My Drive, then run this cell again.")
try:
    import xgboost
    if int(xgboost.__version__.split(".")[0]) < 2:
        raise ImportError
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"], check=True)
os.environ["AMR_AL_ROOT"] = ROOT
sys.path.insert(0, ROOT)
for m in [k for k in sys.modules if k.startswith("amr_al")]:
    del sys.modules[m]                      # always load the latest code from Drive
from amr_al import config, paths, data, models, runner, audit, analysis, driams_fetch
paths.ensure_dirs()
print("shared folder:", ROOT)
print("compute device for XGBoost:", models.device())

## 1 · GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("XGBoost", xgboost.__version__, "| device:", models.device())
if models.device() != "cuda":
    print("\n!! Not on a GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run all.")

## 2 · End-to-end self-test on synthetic data (expect `17/17 checks passed`)

In [ ]:
from amr_al import selftest
ok = selftest.run()

## 3 · T4 speed test: one model fit on spectrum-sized data

In [ ]:
speed = audit.benchmark()
print(speed)

## 4 · Check this account can write to the shared folder

In [ ]:
ACCOUNT = "acc1"          # <- a short name for THIS Colab account (acc1, acc2, ...)
import time, glob
with open(os.path.join(paths.LOGS, f"hello_{ACCOUNT}.txt"), "w") as f:
    f.write(f"{ACCOUNT} {time.ctime()} device={models.device()} speed={speed}\n")
print("accounts seen so far:", sorted(os.path.basename(p)[6:-4] for p in glob.glob(os.path.join(paths.LOGS, "hello_*.txt"))))

**Record:** the self-test summary line, the speed test lines, and anything marked FAIL.